# Control (LCS) Sample — New-Sample Test Harness

Interactive workbench for testing how `LCSDetector`
(`src/detectors/control_detector.py`) and the drift-history chart generator
(`src/visualisers/control_visualize.py`) react to a single new Control
(LCS) sample observation, against a controlled, repeatable historic
baseline.

Two building blocks, defined below:

- **`refill_history(rows)`** — (re)writes a known baseline set of historic
  observations to a **dedicated test history file**, completely replacing
  whatever was there before. This notebook never reads from or writes to
  the real production history
  (`data/processed/lcs_standard_history.csv`) — it uses its own file under
  `data/processed/lcs_test_harness/`, so re-running cells is always safe
  and repeatable.
- **`throw_in_sample(...)`** — runs one new observation through the real
  detector against whatever is currently in that test history file, prints
  the resulting drift status/severity, and renders its updated
  drift-history chart inline.

Typical loop:
`refill_history(...)` (reset to a clean baseline) → `throw_in_sample(...)`
(try a new value) → inspect the printed result + chart → either
`throw_in_sample(...)` again with a different value to keep piling onto the
SAME history (useful for testing a run of several samples / a trend), or
`refill_history(...)` again first to reset back to a clean baseline before
trying an unrelated scenario.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")  # repo root, so `from src...` imports work from notebooks/

import pandas as pd
import yaml
from IPython.display import Image, display

from src.detectors.control_detector import LCSDetector, _ensure_history_columns, _transform_lcs_values
from src.visualisers.control_visualize import generate_control_plots

## Test harness configuration

A dedicated history CSV and config file, isolated from the real
`config/lcs_config.yaml` / `data/processed/lcs_standard_history.csv` used
in production. All other detection settings (thresholds, windows, etc.)
are copied from the real config, so this harness behaves exactly like
production — only *where the history is stored* differs.

In [ ]:
TEST_DIR = Path("../data/processed/lcs_test_harness")
TEST_DIR.mkdir(parents=True, exist_ok=True)

TEST_HISTORY_PATH = TEST_DIR / "lcs_standard_history_TEST.csv"
TEST_PLOTS_DIR = TEST_DIR / "plots"

with open("../config/lcs_config.yaml") as f:
    _base_config = yaml.safe_load(f)
_base_config["history_path"] = str(TEST_HISTORY_PATH)

TEST_CONFIG_PATH = TEST_DIR / "lcs_config_TEST.yaml"
TEST_CONFIG_PATH.write_text(yaml.dump(_base_config))

detector = LCSDetector(str(TEST_CONFIG_PATH))

print("Test history file:", TEST_HISTORY_PATH.resolve())
print("Test plots folder:", TEST_PLOTS_DIR.resolve())

## `refill_history` — reset the historic set to a known baseline

`make_baseline_rows(...)` builds a small, realistic synthetic series for
one `(ANALYTE_CODE, STD_CODE, SCHEME_CODE)` group; `refill_history(...)`
writes it to `TEST_HISTORY_PATH`, fully overwriting anything already
there. Default `n=10` comfortably covers the real config's
`min_history_point` (3) and `min_history_trend` (6), so a freshly refilled
baseline is immediately ready for full point + trend classification.

In [ ]:
def make_baseline_rows(analyte, std_code, scheme_code, *, target, max_v, min_v,
                        max_warn, min_warn, n=10, start_date="2024-01-01",
                        freq_days=3, values=None):
    """
    Build `n` synthetic historic rows for one
    (ANALYTE_CODE, STD_CODE, SCHEME_CODE) group, evenly spaced `freq_days`
    apart starting at `start_date`.

    Pass `values` (a list of length n) for full control over the baseline's
    shape (e.g. a slow drift); omit it to default to n values sitting
    exactly on target (a flat, healthy baseline).
    """
    dates = pd.date_range(start_date, periods=n, freq=f"{freq_days}D")
    if values is None:
        values = [target] * n
    assert len(values) == n, "values must have exactly n entries"

    return [
        {
            "ANALYTICAL_TYPE": "Standard",
            "STD_LOT_CODE": "Sample",
            "STD_CODE": std_code,
            "SCHEME_CODE": scheme_code,
            "JOB_CODE": f"BASELINE_{i}",
            "ANALYTE_CODE": analyte,
            "ANALYSED_DATE": date,
            "NUMERIC_FINAL_VALUE": value,
            "INTERNAL_TARGET_VALUE": target,
            "INTERNAL_MAX_VALUE": max_v,
            "INTERNAL_MIN_VALUE": min_v,
            "INTERNAL_MAX_WARNING_VALUE": max_warn,
            "INTERNAL_MIN_WARNING_VALUE": min_warn,
            "INTERNAL_MAX_INCLUSIVE": "Y",
            "INTERNAL_MIN_INCLUSIVE": "Y",
            "INTERNAL_MAX_WARNING_INCLUSIVE": "Y",
            "INTERNAL_MIN_WARNING_INCLUSIVE": "Y",
            "UNIT_CODE": "MG_KG",
            "STANDARD_STATUS": "Pass",
        }
        for i, (date, value) in enumerate(zip(dates, values), start=1)
    ]


def refill_history(rows, history_path=TEST_HISTORY_PATH):
    """
    Overwrite the test history file with exactly `rows` (a list of raw,
    untransformed observation dicts -- see make_baseline_rows()). Fully
    replaces whatever was there before, so every call starts the next test
    from a clean, known baseline instead of accumulating across notebook
    re-runs.
    """
    raw = pd.DataFrame(rows)
    history = _ensure_history_columns(_transform_lcs_values(raw))
    history.to_csv(history_path, index=False)

    n_groups = history[["ANALYTE_CODE", "STD_CODE", "SCHEME_CODE"]].drop_duplicates().shape[0]
    print(f"Refilled {history_path.name} with {len(history)} historic row(s) across {n_groups} group(s).")

## `throw_in_sample` — run one new observation through the detector

Builds a single-row observation, runs it through `detector.detect()`
against whatever is currently persisted in the test history file, prints
the resulting `DRIFT_STATUS` / `DRIFT_SEVERITY` / reason, and (by default)
renders + displays its drift-history chart inline via
`generate_control_plots(..., only_flagged=False)` — unlike production use,
this harness always wants to see the chart, even for a perfectly normal
sample.

In [ ]:
def throw_in_sample(*, analyte, std_code, scheme_code, value, analysed_date,
                     target, max_v, min_v, max_warn, min_warn,
                     job_code="NEW_SAMPLE", detector=detector, show_chart=True):
    """
    Run ONE new observation through `detector.detect()` against whatever is
    currently persisted in the test history file, print the outcome, and
    (by default) render + display its updated drift-history chart inline.

    Returns the LCSAnomalyResult from this call.
    """
    new_row = pd.DataFrame([{
        "ANALYTICAL_TYPE": "Standard",
        "STD_LOT_CODE": "Sample",
        "STD_CODE": std_code,
        "SCHEME_CODE": scheme_code,
        "JOB_CODE": job_code,
        "ANALYTE_CODE": analyte,
        "ANALYSED_DATE": pd.Timestamp(analysed_date),
        "NUMERIC_FINAL_VALUE": value,
        "INTERNAL_TARGET_VALUE": target,
        "INTERNAL_MAX_VALUE": max_v,
        "INTERNAL_MIN_VALUE": min_v,
        "INTERNAL_MAX_WARNING_VALUE": max_warn,
        "INTERNAL_MIN_WARNING_VALUE": min_warn,
        "INTERNAL_MAX_INCLUSIVE": "Y",
        "INTERNAL_MIN_INCLUSIVE": "Y",
        "INTERNAL_MAX_WARNING_INCLUSIVE": "Y",
        "INTERNAL_MIN_WARNING_INCLUSIVE": "Y",
        "UNIT_CODE": "MG_KG",
        "STANDARD_STATUS": "Pass",
    }])

    result = detector.detect(new_row)
    matches = [r for r in result.details["results"] if r["ANALYTE_CODE"] == analyte]
    outcome = matches[-1] if matches else None

    print(f"detected={result.detected}  severity={result.severity}  confidence={result.confidence:.0f}%")
    if outcome:
        print(f"OFFSET={outcome['OFFSET']:+.4f}  N_HISTORY={outcome['N_HISTORY']}  "
              f"DRIFT_STATUS={outcome['DRIFT_STATUS']}  DRIFT_SEVERITY={outcome['DRIFT_SEVERITY']}")
        print(f"Reason: {outcome['DRIFT_REASON']}")
    else:
        print("No result row found for this analyte (unexpected).")

    if show_chart and outcome:
        plots = generate_control_plots(result, detector.history_path, TEST_PLOTS_DIR, only_flagged=False)
        path = plots.get((analyte, std_code, scheme_code))
        if path:
            display(Image(filename=str(path)))
        else:
            print("(no chart generated for this group)")

    return result

## Example: refill the baseline, then throw in a new sample

Baseline values below (target/limits) are drawn from a real Cu / OREAS_502C
/ GE_ICP40Q12 Control sample row (see `data/samples/CONTROL_FAIL_1.csv`),
so the scale is realistic rather than an arbitrary round number.

In [ ]:
CU_GROUP = dict(analyte="CU", std_code="OREAS_502C", scheme_code="GE_ICP40Q12")
CU_LIMITS = dict(target=7830.0, max_v=8614.566, min_v=7045.434, max_warn=8327.988, min_warn=7332.012)

refill_history(make_baseline_rows(**CU_GROUP, **CU_LIMITS, n=10, start_date="2024-01-01", freq_days=3))

In [ ]:
# A clear upper-failure breach (real value from CONTROL_FAIL_1.csv).
result = throw_in_sample(**CU_GROUP, **CU_LIMITS, value=8912.7, analysed_date="2024-01-31")

## Test it again: refill, then try a different new sample

This is the repeatable loop the harness is built for: reset to the exact
same clean baseline, then try a different new observation — here, a
perfectly normal one — without any leftover state from the run above.

In [ ]:
refill_history(make_baseline_rows(**CU_GROUP, **CU_LIMITS, n=10, start_date="2024-01-01", freq_days=3))

# A normal, healthy sample this time.
result = throw_in_sample(**CU_GROUP, **CU_LIMITS, value=7845.0, analysed_date="2024-01-31")

## Peek at the current test history

Useful for sanity-checking what `refill_history()` / `throw_in_sample()`
have actually persisted so far.

In [ ]:
pd.read_csv(TEST_HISTORY_PATH, parse_dates=["ANALYSED_DATE"])[
    ["ANALYTE_CODE", "STD_CODE", "SCHEME_CODE", "ANALYSED_DATE", "NUMERIC_FINAL_VALUE", "TRANSFORMED_VALUE"]
]

## Notes

- **Isolation**: this notebook only ever touches
  `data/processed/lcs_test_harness/` — it never reads from or writes to the
  real `data/processed/lcs_standard_history.csv`.
- **Accumulation vs. reset**: calling `throw_in_sample(...)` repeatedly
  *without* calling `refill_history(...)` in between keeps adding to the
  SAME test history (mirroring real production behaviour) — handy for
  testing a run of several samples in a row, or a developing trend. Call
  `refill_history(...)` whenever you want a clean slate again.
- **History-depth gating**: `LCSDetector` requires at least
  `min_history_point` (3 by default) prior observations before it reports
  anything other than `INSUFFICIENT_HISTORY`/`NONE`, no matter how extreme
  a new value is — keep `make_baseline_rows(n=...)` at or above that (the
  default `n=10` already clears both `min_history_point` and
  `min_history_trend`).
- **Automated equivalent**: see `tests/test_control_detector.py` and
  `tests/test_control_visualize.py` for the same kind of scenario expressed
  as pytest assertions rather than interactive prints/charts.

## Generate real Control-analyte demo charts for the POC (`poc/images/`)

The proof of concept's Control detail page (`poc/index.html` → … →
`poc/detail.html?detector=control`) originally showed 5 hand-drawn placeholder
SVGs with invented metric text. This section replaces them with the real
thing: 5 different `ANALYTE_CODE`s (Cu, Zn, Pb, Ni, Fe), each seeded with a
baseline + a "current" sample crafted to reproduce the exact severity story
`poc/data.js` already narrates for its dummy items -- verified against the
real `LCSDetector` classification below, not assumed -- then rendered via
the real chart generator (`src/visualisers/control_visualize.py`) straight
into `poc/images/`.

This reuses every function already defined above (`make_baseline_rows`,
`refill_history`, the shared `detector`, `generate_control_plots`) -- no new
detection or plotting logic, only orchestration.

In [ ]:
# Shared target/limits across all 5 demo analytes (real values from a Cu /
# OREAS_502C / GE_ICP40Q12 row, data/samples/CONTROL_FAIL_1.csv -- reused
# for every analyte here since exact chemistry doesn't matter for this demo,
# only the resulting drift pattern/severity does).
DEMO_LIMITS = dict(target=7830.0, max_v=8614.566, min_v=7045.434, max_warn=8327.988, min_warn=7332.012)
DEMO_STD_CODE = "OREAS_502C"
DEMO_SCHEME_CODE = "GE_ICP40Q12"

# Each analyte's baseline shape + new-sample value, crafted to reproduce
# EXACTLY the severity story poc/data.js already narrates for its dummy
# items -- confirmed against the real LCSDetector's actual classification
# (see the printed output two cells down), not assumed:
#   CU -> a genuine point breach                       -> CRITICAL
#   ZN -> a strong, consistent upward trend escalated
#         to *_FAILURE_DRIFT                            -> HIGH
#   PB -> the mirrored downward-trend case               -> HIGH
#   NI -> sitting directly in the upper warning band     -> MEDIUM
#   FE -> flat and healthy                               -> NONE
DEMO_ANALYTES = {
    "CU": {"baseline": [7830.0] * 10, "new_value": 8912.7},
    "ZN": {"baseline": [7830.0 + i * 41 for i in range(10)], "new_value": 8262.0},
    "PB": {"baseline": [7830.0 - i * 41 for i in range(10)], "new_value": 7398.0},
    "NI": {"baseline": [7830.0] * 10, "new_value": 8400.0},
    "FE": {"baseline": [7830.0] * 10, "new_value": 7845.0},
}

all_baseline_rows = []
for analyte, cfg in DEMO_ANALYTES.items():
    all_baseline_rows += make_baseline_rows(
        analyte, DEMO_STD_CODE, DEMO_SCHEME_CODE, **DEMO_LIMITS,
        n=10, start_date="2024-01-01", freq_days=3, values=cfg["baseline"],
    )

# refill_history() fully overwrites the file -- seeding all 5 groups at once
# (rather than 5 separate calls) is what "update it... but for 5 different
# ANALYTE_CODEs" means in practice.
refill_history(all_baseline_rows)

In [ ]:
# One combined "current observation" per analyte, run through detect() in a
# single call -- this produces ONE LCSAnomalyResult whose
# details["results"] holds all 5 groups' latest classification at once.
new_sample_rows = pd.DataFrame([
    {
        "ANALYTICAL_TYPE": "Standard",
        "STD_LOT_CODE": "Sample",
        "STD_CODE": DEMO_STD_CODE,
        "SCHEME_CODE": DEMO_SCHEME_CODE,
        "JOB_CODE": "NEW_SAMPLE",
        "ANALYTE_CODE": analyte,
        "ANALYSED_DATE": pd.Timestamp("2024-01-31"),
        "NUMERIC_FINAL_VALUE": cfg["new_value"],
        "INTERNAL_TARGET_VALUE": DEMO_LIMITS["target"],
        "INTERNAL_MAX_VALUE": DEMO_LIMITS["max_v"],
        "INTERNAL_MIN_VALUE": DEMO_LIMITS["min_v"],
        "INTERNAL_MAX_WARNING_VALUE": DEMO_LIMITS["max_warn"],
        "INTERNAL_MIN_WARNING_VALUE": DEMO_LIMITS["min_warn"],
        "INTERNAL_MAX_INCLUSIVE": "Y",
        "INTERNAL_MIN_INCLUSIVE": "Y",
        "INTERNAL_MAX_WARNING_INCLUSIVE": "Y",
        "INTERNAL_MIN_WARNING_INCLUSIVE": "Y",
        "UNIT_CODE": "MG_KG",
        "STANDARD_STATUS": "Pass",
    }
    for analyte, cfg in DEMO_ANALYTES.items()
])

demo_result = detector.detect(new_sample_rows)

# _SEVERITY_CONFIDENCE mirrors control_detector.py's own module-level
# mapping -- LCSAnomalyResult.confidence is a single figure for the WHOLE
# (multi-analyte) result, so per-analyte confidence for poc/data.js's
# metrics panel is derived the same way the detector itself would.
_SEVERITY_CONFIDENCE = {"CRITICAL": 100.0, "HIGH": 85.0, "MEDIUM": 65.0, "NONE": 95.0}

for r in demo_result.details["results"]:
    confidence = _SEVERITY_CONFIDENCE.get(r["DRIFT_SEVERITY"], 50.0)
    print(f"{r['ANALYTE_CODE']}: OFFSET={r['OFFSET']:+.4f}  SEVERITY={r['DRIFT_SEVERITY']}  "
          f"CONFIDENCE={confidence:.0f}%  STATUS={r['DRIFT_STATUS']}")
    print(f"   {r['DRIFT_REASON']}")

In [ ]:
# Render the real charts straight into the POC's images folder.
# only_flagged=False so FE's NONE-severity chart still renders -- a demo
# needs to show the "healthy" case too, not just the anomalous ones.
POC_IMAGES_DIR = Path("../poc/images")

demo_plots = generate_control_plots(demo_result, detector.history_path, POC_IMAGES_DIR, only_flagged=False)

for group_key, path in sorted(demo_plots.items()):
    print(group_key, "->", path.name)

**Next step (manual, outside this notebook):** update `poc/data.js`'s
`control` items with the exact `imagePath` filenames and
OFFSET/CONFIDENCE/DRIFT_REASON values printed above, and delete the 5
now-superseded hand-drawn SVGs (`poc/images/{cu,zn,pb,ni,fe}.svg`).